# CIS 5300 Milestone 2: cartoon-caption baselines



In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

drive.mount('/content/drive')
DRIVE_RUN = Path('/content/drive/MyDrive/cis5300_milestone2_qwen35_4b')
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
print('Upload milestone2_colab_bundle.zip from the prepared project folder.')
uploaded = files.upload()
archive = next((name for name in uploaded if name.endswith('.zip')), None)
assert archive is not None, 'Upload the prepared ZIP bundle.'
ROOT = Path('/content/cis5300_m2')
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        target = (ROOT / item.filename).resolve()
        assert target.is_relative_to(ROOT.resolve()), 'Unsafe archive path'
    z.extractall(ROOT)
del uploaded
PROJECT = ROOT / 'milestone2'
os.chdir(PROJECT)
# Preserve bundled local output; use Drive for durable run output.
local_output = PROJECT / 'output'
if not local_output.is_symlink():
    if local_output.exists():
        bundled = PROJECT / 'bundled-output'
        if not bundled.exists():
            local_output.rename(bundled)
        else:
            raise RuntimeError('Unexpected duplicate output directory; inspect before continuing.')
    local_output.symlink_to(DRIVE_RUN, target_is_directory=True)

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)
print('Working directory:', PROJECT)
print('Durable output:', DRIVE_RUN)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Upload milestone2_colab_bundle.zip from the prepared project folder.


Saving milestone2_colab_bundle.zip to milestone2_colab_bundle.zip
Working directory: /content/cis5300_m2/milestone2
Durable output: /content/drive/MyDrive/cis5300_milestone2_qwen35_4b


In [ ]:
run('-m', 'pip', 'install', '-U', '-r', 'requirements-gpu.txt')
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first.'
print(torch.cuda.get_device_name(0))
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
run('audit-data.py')
run('-m', 'unittest', '-v', 'test_evaluation.py')
run('run.py')


NVIDIA L4
GPU memory GB: 22.0


## Dev-only smoke test



In [ ]:
for task in ('matching', 'ranking'):
    run('strong-baseline.py', '--task', task,
        '--input', f'../milestone1/data/{task}/dev.jsonl',
        '--data-root', '../milestone1/data',
        '--output', f'output/smoke_{task}.jsonl', '--limit', '8')
    rows = [json.loads(line) for line in Path(f'output/smoke_{task}.jsonl').read_text().splitlines()]
    print(task, rows)
    assert any(r['prediction'] is not None for r in rows), 'All outputs invalid: inspect model formatting.'


matching [{'instance_id': '2ecf456fa37b3e6bf94ed1e28f959dc7', 'prediction': 'D', 'raw_output': 'D'}, {'instance_id': '2b1f8c98445baf255e3fca9edb086367', 'prediction': 'D', 'raw_output': 'D'}, {'instance_id': '26ae0cb3e4fe83b9565e6d2b380635aa', 'prediction': 'C', 'raw_output': 'C'}, {'instance_id': 'df528bb5206aa14f25d1109e00c6d56b', 'prediction': 'D', 'raw_output': 'D'}, {'instance_id': '8355b327d02ca1bc10f798eccc3b64b8', 'prediction': 'C', 'raw_output': 'C'}, {'instance_id': '174c3517748a00f9f22a09cfca6cafe0', 'prediction': 'B', 'raw_output': 'B'}, {'instance_id': 'd021c8d40bc6595409a807f1c8fea66c', 'prediction': 'B', 'raw_output': 'B'}, {'instance_id': 'f22d296e8dffc637181c4c9a2937c4de', 'prediction': 'E', 'raw_output': 'E'}]
ranking [{'pair_id': '536_easy_0', 'prediction': 'B', 'raw_output': 'B'}, {'pair_id': '536_easy_1', 'prediction': 'B', 'raw_output': 'B'}, {'pair_id': '536_easy_2', 'prediction': 'B', 'raw_output': 'B'}, {'pair_id': '536_easy_3', 'prediction': 'B', 'raw_output':

## Full dev evaluation


In [ ]:
run('run.py', '--strong', '--split', 'dev')
meta = [json.loads(Path(f'output/strong_pixels_{task}_dev.jsonl.meta.json').read_text())
        for task in ('matching', 'ranking')]
assert meta[0]['resolved_revision'] == meta[1]['resolved_revision']
REVISION = meta[0]['resolved_revision']
print('Frozen model revision:', REVISION)
for task in ('matching', 'ranking'):
    print(task, Path(f'output/strong_pixels_{task}_dev.jsonl.scores.json').read_text())


Frozen model revision: 851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
matching {
  "task": "matching",
  "overall": {
    "n": 531,
    "correct": 251,
    "accuracy": 0.4726930320150659,
    "accuracy_percent": 47.26930320150659,
    "invalid": 0
  }
}

ranking {
  "task": "ranking",
  "overall": {
    "n": 880,
    "correct": 447,
    "accuracy": 0.5079545454545454,
    "accuracy_percent": 50.79545454545455,
    "invalid": 0
  },
  "by_tier": {
    "easy": {
      "n": 440,
      "correct": 213,
      "accuracy": 0.48409090909090907,
      "accuracy_percent": 48.40909090909091,
      "invalid": 0
    },
    "hard": {
      "n": 440,
      "correct": 234,
      "accuracy": 0.5318181818181819,
      "accuracy_percent": 53.18181818181818,
      "invalid": 0
    }
  }
}



### Frozen test evaluation


In [ ]:
run('run.py', '--strong', '--split', 'test', '--revision', REVISION)
run('make-report.py', '--require-complete')
for task in ('matching', 'ranking'):
    print(task, Path(f'output/strong_pixels_{task}_test.jsonl.scores.json').read_text())


matching {
  "task": "matching",
  "overall": {
    "n": 528,
    "correct": 262,
    "accuracy": 0.4962121212121212,
    "accuracy_percent": 49.621212121212125,
    "invalid": 0
  }
}

ranking {
  "task": "ranking",
  "overall": {
    "n": 940,
    "correct": 484,
    "accuracy": 0.5148936170212766,
    "accuracy_percent": 51.48936170212766,
    "invalid": 0
  },
  "by_tier": {
    "easy": {
      "n": 470,
      "correct": 248,
      "accuracy": 0.5276595744680851,
      "accuracy_percent": 52.765957446808514,
      "invalid": 0
    },
    "hard": {
      "n": 470,
      "correct": 236,
      "accuracy": 0.502127659574468,
      "accuracy_percent": 50.212765957446805,
      "invalid": 0
    }
  }
}



In [ ]:
from IPython.display import display, IFrame
report = Path('output/pdf/milestone2-report.pdf')
assert report.exists()
display(IFrame(str(report), width=850, height=700))
results_zip = Path('/content/milestone2_gpu_results.zip')
with zipfile.ZipFile(results_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in DRIVE_RUN.rglob('*'):
        if path.is_file() and path.name != 'milestone2-draft.pdf':
            z.write(path, Path('output') / path.relative_to(DRIVE_RUN))
files.download(str(results_zip))
files.download(str(report))


In [17]:
from google.colab import files
from pathlib import Path
import os

os.chdir('/content/cis5300_m2/milestone2')

uploaded = files.upload()
Path('order-swap.py').write_bytes(uploaded['order-swap.py'])

%run order-swap.py --run

files.download('output/order-swap-dev/order-swap-results.zip')

Saving order-swap.py to order-swap.py
Running 880 swapped DEV examples. Original predictions are reused.

Loading weights: 100%|██████████| 723/723 [00:02<00:00, 321.22it/s]
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is fallin

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>